# Day 02：非线性激活函数 —— Sigmoid 与 ReLU

> ☀️ 第二周 · 破局与复兴 · 第 2 天

昨天我们用隐藏层解决了 XOR，但用的阶跃函数有一个致命缺陷：**梯度几乎处处为零**。

梯度为零意味着无法用梯度下降来训练网络——权重不知道该往哪个方向调整。

今天，我们要学习阶跃函数的替代品：**Sigmoid** 和 **ReLU**。它们让神经网络从「能用」变成「能训练」。

**今天的任务**：理解激活函数为什么必须是非线性的，以及 Sigmoid/ReLU 各自的优缺点。

---
## 1. 历史背景：从硬决策到软决策

### 阶跃函数的辉煌与局限

1943 年，McCulloch 和 Pitts 提出的最早的神经元模型就使用了阶跃函数——大于阈值就「激发」，否则就「沉默」。这很符合生物神经元的「全有或全无」特性。

但当我们想要**训练**网络时，问题来了：

想象你在山上蒙着眼睛找最低点。你的策略是「感受脚下的坡度，往低处走」。但如果地面是完全平的（梯度为零），你根本感觉不到该往哪走——这就是阶跃函数的困境。

### Sigmoid 的登场

1986 年，Rumelhart 等人在反向传播论文中使用了 **Sigmoid 函数**作为激活函数。它是一条光滑的 S 形曲线，处处可导，梯度非零。

Sigmoid 让反向传播算法得以实际运作，开启了神经网络的复兴。

### ReLU 的崛起

2010 年前后，研究者发现 Sigmoid 在深层网络中会导致**梯度消失**问题。Nair 和 Hinton 在 2010 年的工作表明，简单的 **ReLU（Rectified Linear Unit）** 在深层网络中表现更好。

ReLU 只需要一个 `max(0, z)` 运算，计算极简，却解决了梯度消失问题。它迅速成为深度学习的默认激活函数，至今仍被广泛使用。

---
## 2. 生活隐喻：裁判的三种风格

把激活函数想象成三种不同风格的裁判，面对同一个问题：「这个球员犯规了吗？」

### 阶跃函数裁判（非黑即白）
- 规则：只要碰到了就判犯规，否则不判
- 没有灰色地带，没有「轻微接触」
- 问题：判罚后你问他「为什么？」，他说不出来（梯度为零）

### Sigmoid 裁判（概率思维）
- 规则：根据接触程度给出犯规概率，比如「70% 可能犯规」
- 有中间地带，能表达不确定性
- 能解释判罚理由（梯度非零），但理由在极端情况下变得模糊（梯度消失）

### ReLU 裁判（睁一只眼闭一只眼）
- 规则：有明显接触就按接触力度判，轻微接触直接忽略
- 简单高效，但对「轻微犯规」完全视而不见（负区间梯度为零）
- 现代足球 VAR 系统的风格——抓大放小

---
## 3. 数学直觉：激活函数详解

### 为什么需要激活函数？（非线性的必要性）

在讲具体函数之前，先回答一个根本问题：**为什么必须有激活函数？**

假设没有激活函数，一个两层网络的计算是：

$$y = W_2 \cdot (W_1 \cdot x + b_1) + b_2 = (W_2 W_1) \cdot x + (W_2 b_1 + b_2)$$

这等价于 $y = W' \cdot x + b'$——**还是一个单层线性变换！**

矩阵乘法满足结合律，所以无论叠多少层，没有激活函数的多层网络**等价于单层网络**。

激活函数的作用就是在每层之间插入一个**非线性变换**，打破这个等价关系，让多层网络真正拥有更强的表达能力。

就像做菜：食材（数据）经过厨刀（线性变换）处理后，必须经过**烹饪**（非线性激活）才能变成菜肴。生的食材堆再多层也还是生的。

### 阶跃函数的梯度困境

昨天我们用的阶跃函数：

$$f(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases}$$

它的导数：

$$f'(z) = \begin{cases} 0 & \text{if } z \neq 0 \\ \text{未定义} & \text{if } z = 0 \end{cases}$$

梯度为零意味着：**误差信号无法从输出层传回隐藏层**。反向传播的第一步就卡住了——我们不知道每个权重对最终误差的贡献有多大。

用代码直观感受一下这个问题：

In [ ]:
import torch
import matplotlib.pyplot as plt

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# 阶跃函数及其梯度
def step(z):
    """阶跃函数：大于0输出1，否则0"""
    return (z > 0).float()

# 生成输入范围
z = torch.linspace(-5, 5, 200)  # 200个点从-5到5

# 画阶跃函数
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)  # 左图：函数本身
plt.plot(z.numpy(), step(z).numpy(), 'b-', linewidth=2)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('输入 z')
plt.ylabel('输出 f(z)')
plt.title('阶跃函数')
plt.ylim(-0.2, 1.2)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)  # 右图：梯度（几乎全为0）
grad = torch.zeros_like(z)  # 梯度全是0
plt.plot(z.numpy(), grad.numpy(), 'r-', linewidth=2)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('输入 z')
plt.ylabel('梯度 f\'(z)')
plt.title('阶跃函数的梯度：几乎处处为零！')
plt.ylim(-0.1, 0.3)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('step_function_gradient.png', dpi=150, bbox_inches='tight')
plt.show()

print("左图：阶跃函数——非黑即白")
print("右图：梯度——除了 z=0 处，其他全是 0")
print("梯度为 0 = 无法学习！")

### Sigmoid：柔和的 S 曲线

Sigmoid 函数是阶跃函数的「柔和版」：

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

关键特性：
- **输出范围 (0, 1)**：可以理解为概率——0.7 表示「70% 倾向于正类」
- **处处可导**：在任何点都有非零梯度
- **在 z=0 附近敏感**：输入的小变化会引起输出的明显变化
- **在两端饱和**：当 z 很大或很小时，输出几乎不变（梯度趋近于零）

Sigmoid 的导数有一个优美的性质：

$$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

不需要复杂的求导——只要知道函数值就能算出梯度！

用代码看看 Sigmoid 的形状和梯度：

In [ ]:
import torch
import matplotlib.pyplot as plt

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# Sigmoid 函数定义
def sigmoid(z):
    """Sigmoid 函数：将任意实数映射到 (0, 1) 区间"""
    return 1 / (1 + torch.exp(-z))  # σ(z) = 1/(1+e^(-z))

def sigmoid_grad(z):
    """Sigmoid 的导数：σ'(z) = σ(z) * (1 - σ(z))"""
    s = sigmoid(z)  # 先算函数值
    return s * (1 - s)  # 导数 = 函数值 * (1 - 函数值)

# 生成输入
z = torch.linspace(-6, 6, 200)

# 画图
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)  # Sigmoid 函数
plt.plot(z.numpy(), sigmoid(z).numpy(), 'b-', linewidth=2)
plt.axhline(y=0.5, color='green', linestyle=':', alpha=0.5, label='y=0.5')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('输入 z')
plt.ylabel('σ(z)')
plt.title('Sigmoid 函数')
plt.ylim(-0.1, 1.1)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)  # Sigmoid 导数
plt.plot(z.numpy(), sigmoid_grad(z).numpy(), 'r-', linewidth=2)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('输入 z')
plt.ylabel('σ\'(z)')
plt.title('Sigmoid 导数：中间大，两端小')
plt.ylim(-0.05, 0.3)
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)  # 阶跃 vs Sigmoid 对比
step_y = (z > 0).float()
plt.plot(z.numpy(), step_y.numpy(), 'b--', linewidth=2, label='阶跃函数', alpha=0.7)
plt.plot(z.numpy(), sigmoid(z).numpy(), 'r-', linewidth=2, label='Sigmoid')
plt.xlabel('输入 z')
plt.ylabel('输出')
plt.title('阶跃 vs Sigmoid：从硬到软')
plt.legend()
plt.ylim(-0.1, 1.1)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sigmoid_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sigmoid 的特点：")
print("  - z=0 时输出 0.5（最大不确定性）")
print("  - z 很大时输出趋近 1，z 很小时趋近 0")
print("  - 导数在 z=0 处最大（0.25），两端趋近 0（梯度消失！）")

### Sigmoid 的梯度消失问题

Sigmoid 解决了阶跃函数「梯度全为零」的问题，但它带来了新问题：**梯度消失（Vanishing Gradient）**。

看导数的图：最大值只有 0.25（在 z=0 处）。当 z 很大或很小时，梯度趋近于 0。

在反向传播中，梯度是逐层相乘的。如果每层的梯度都小于 1，经过多层相乘后：

$$0.25 \times 0.25 \times 0.25 \times \cdots \rightarrow 0$$

越靠近输入层的权重，收到的梯度信号越微弱——就像传话游戏，传了几轮之后信息就完全丢失了。

这是**梯度消失**第一次登场。它会在后面的课程中反复出现，是深度学习的核心挑战之一。

### ReLU：简单高效的现代选择

ReLU（Rectified Linear Unit，修正线性单元）的公式极其简单：

$$ReLU(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases}$$

它的导数同样简单：

$$ReLU'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases}$$

ReLU 的优势：
- **梯度不消失**：z > 0 时梯度恒为 1，不会逐层衰减
- **计算极简**：只需要比较和取最大值，比 Sigmoid 的指数运算快得多
- **稀疏激活**：负输入直接输出 0，让部分神经元「沉默」，提高效率

ReLU 的缺点：
- **Dead ReLU**：如果一个神经元的输入永远为负，它就永远输出 0，梯度也永远为 0，这个神经元就「死了」
- **在 z=0 处不可导**：不过实际中这个概率极低，影响不大

用代码看看 ReLU 的形状：

In [ ]:
import torch
import matplotlib.pyplot as plt

# 中文字体配置
try:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Helvetica Neue', 'Heiti SC']
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

# ReLU 函数定义
def relu(z):
    """ReLU 函数：负数输出0，正数输出原值"""
    return torch.maximum(z, torch.zeros_like(z))  # max(0, z)

def relu_grad(z):
    """ReLU 的导数：正数区间梯度为1，负数区间为0"""
    return (z > 0).float()  # z>0 时为 1，否则为 0

# Sigmoid（用于对比）
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))

# 生成输入
z = torch.linspace(-5, 5, 200)

# 画图
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)  # ReLU 函数
plt.plot(z.numpy(), relu(z).numpy(), 'g-', linewidth=2)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('输入 z')
plt.ylabel('ReLU(z)')
plt.title('ReLU 函数')
plt.ylim(-0.5, 5.5)
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)  # ReLU 导数
plt.plot(z.numpy(), relu_grad(z).numpy(), 'r-', linewidth=2)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('输入 z')
plt.ylabel('ReLU\'(z)')
plt.title('ReLU 导数：z>0 时梯度恒为 1')
plt.ylim(-0.1, 1.5)
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)  # 三种激活函数对比
step_y = (z > 0).float()
plt.plot(z.numpy(), step_y.numpy(), 'b--', linewidth=2, label='阶跃', alpha=0.7)
plt.plot(z.numpy(), sigmoid(z).numpy(), 'r-', linewidth=2, label='Sigmoid')
plt.plot(z.numpy(), relu(z).numpy(), 'g-', linewidth=2, label='ReLU')
plt.xlabel('输入 z')
plt.ylabel('输出')
plt.title('三种激活函数对比')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('relu_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("ReLU 的特点：")
print("  - z > 0 时就是一条斜线（y = z），梯度恒为 1")
print("  - z ≤ 0 时直接输出 0，梯度也为 0")
print("  - 比 Sigmoid 简单得多，没有指数运算！")

### 四种激活函数总结

| 激活函数 | 公式 | 输出范围 | 导数 | 优点 | 缺点 |
|----------|------|----------|------|------|------|
| 阶跃 | 1 if z>0 else 0 | {0, 1} | 几乎处处 0 | 简单 | 无法训练 |
| Sigmoid | 1/(1+e^(-z)) | (0, 1) | σ(z)(1-σ(z)) | 处处可导，输出是概率 | 梯度消失 |
| ReLU | max(0, z) | [0, +∞) | 1 if z>0 else 0 | 计算快，梯度不消失 | 神经元死亡 |
| Tanh | (e^z-e^(-z))/(e^z+e^(-z)) | (-1, 1) | 1-tanh²(z) | 居中输出 | 梯度消失 |

**选择指南**：
- 现代深度学习默认用 **ReLU**（隐藏层）
- 二分类任务的输出层用 **Sigmoid**（输出概率）
- 阶跃函数只用于理论分析，不用于实际训练
- Tanh 在 RNN/LSTM 中常用（输出以 0 为中心）

---
## 4. 代码实现：用 Sigmoid 替代阶跃函数

现在把昨天的隐藏层感知机改造一下，用 Sigmoid 替代阶跃函数。

注意权重需要放大（乘以 10），因为 Sigmoid 的「软区间」比阶跃函数宽，需要更大的权重才能模拟出类似阶跃的效果。

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))  # σ(z) = 1/(1+e^(-z))

class MLPWithSigmoid:
    """用 Sigmoid 作为激活函数的多层感知机

    结构：输入层(2) → 隐藏层(2, Sigmoid) → 输出层(1, Sigmoid)
    """

    def __init__(self):
        """初始化权重（放大10倍以模拟阶跃效果）"""
        # 第一层权重：和昨天一样的结构，但权重放大10倍
        # 因为 Sigmoid 在 z=0 附近是「软」的，需要更大的 z 值才能接近 0 或 1
        self.W1 = torch.tensor([[10.0, 10.0], [10.0, 10.0]])  # 隐藏层权重
        self.b1 = torch.tensor([[-5.0], [-15.0]])  # 偏置也放大：OR=-5, AND=-15

        # 第二层权重
        self.W2 = torch.tensor([[10.0], [-10.0]])  # 输出层权重
        self.b2 = torch.tensor([[-5.0]])  # 输出层偏置

    def forward(self, X):
        """前向传播：使用 Sigmoid 激活

        参数:
            X: 输入张量 [batch_size, 2]
        返回:
            输出概率 [batch_size, 1]，值在 (0, 1) 之间
        """
        # 第一层：线性变换 + Sigmoid 激活
        z1 = torch.matmul(X, self.W1.T) + self.b1.T  # 加权求和
        a1 = sigmoid(z1)  # Sigmoid 激活（替代阶跃函数）

        # 第二层：线性变换 + Sigmoid 激活
        z2 = torch.matmul(a1, self.W2) + self.b2  # 加权求和
        a2 = sigmoid(z2)  # 输出层也用 Sigmoid，输出概率

        return a2

# 创建模型
mlp = MLPWithSigmoid()
print("Sigmoid MLP 创建完成")
print("  隐藏层激活：Sigmoid（替代阶跃函数）")
print("  输出层激活：Sigmoid（输出概率 0~1）")

---
## 5. 验证实验

用 Sigmoid MLP 验证 XOR 问题。注意这次输出的是**概率**，不是 0/1 硬判断。

In [ ]:
import torch

def sigmoid(z):
    """Sigmoid 激活函数"""
    return 1 / (1 + torch.exp(-z))

class MLPWithSigmoid:
    """用 Sigmoid 的多层感知机"""
    def __init__(self):
        self.W1 = torch.tensor([[10.0, 10.0], [10.0, 10.0]])  # 隐藏层权重
        self.b1 = torch.tensor([[-5.0], [-15.0]])  # 隐藏层偏置
        self.W2 = torch.tensor([[10.0], [-10.0]])  # 输出层权重
        self.b2 = torch.tensor([[-5.0]])  # 输出层偏置

    def forward(self, X):
        """前向传播"""
        z1 = torch.matmul(X, self.W1.T) + self.b1.T  # 隐藏层线性变换
        a1 = sigmoid(z1)  # Sigmoid 激活
        z2 = torch.matmul(a1, self.W2) + self.b2  # 输出层线性变换
        return sigmoid(z2)  # Sigmoid 输出概率

# XOR 数据
X_xor = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# 前向传播
mlp = MLPWithSigmoid()
y_pred = mlp.forward(X_xor)  # 输出概率

# 验证
print("Sigmoid MLP 验证 XOR")
print("=" * 55)
print(f"{'输入':<12} {'真实标签':<10} {'输出概率':<12} {'预测':<8} {'结果'}")
print("-" * 55)

all_correct = True
for i in range(4):  # 遍历四个输入
    x_str = f"({X_xor[i,0].item()}, {X_xor[i,1].item()})"  # 输入
    true_label = int(y_xor[i].item())  # 真实标签
    prob = y_pred[i].item()  # Sigmoid 输出概率
    pred = 1 if prob > 0.5 else 0  # 大于0.5判为1
    status = "✓ 正确" if true_label == pred else "✗ 错误"
    if true_label != pred:
        all_correct = False
    print(f"{x_str:<12} {true_label:<10} {prob:<12.4f} {pred:<8} {status}")

print("-" * 55)
if all_correct:
    print("全部正确！Sigmoid MLP 同样解决了 XOR！")
print("\n注意：Sigmoid 输出的是概率（0~1），不是硬判断。")
print("例如 0.0007 表示「几乎确定是 0」，0.9993 表示「几乎确定是 1」。")

---
## 6. 翻译词典

| 生活直觉 | 深度学习术语 | 英文 |
|----------|-------------|------|
| 非黑即白的裁判 | 阶跃函数 | Step Function |
| 概率思维的裁判 | Sigmoid 函数 | Sigmoid / Logistic Function |
| 睁一只眼闭一只眼 | ReLU 函数 | Rectified Linear Unit |
| 裁判说不清理由 | 梯度为零 | Zero Gradient |
| 传话越传越模糊 | 梯度消失 | Vanishing Gradient |
| 做菜必须烹饪 | 非线性激活 | Non-linear Activation |
| 犯规概率 | 输出概率 | Output Probability |

---
## 7. 总结与预告

### 今日核心收获

1. **为什么需要激活函数**：没有非线性激活，多层网络等价于单层（矩阵乘法结合律）
2. **阶跃函数的问题**：梯度几乎处处为零，无法用梯度下降训练
3. **Sigmoid**：处处可导，输出可解释为概率，但存在梯度消失问题
4. **ReLU**：计算简单，梯度不消失，是现代深度学习的默认选择
5. **梯度消失第一次登场**：Sigmoid 的最大梯度只有 0.25，多层相乘后趋近于零

### 明天预告

今天我们知道了激活函数让网络「能训练」，但还没有真正**训练**过它。

明天学习**损失函数**和**梯度下降**——如何衡量模型的「错误程度」，以及如何沿着梯度方向一步步走向最优解。